In [0]:
from pyspark.sql import functions as F
import matplotlib.pyplot as plt

#Load Data

In [0]:
TEAM_DIR = "dbfs:/student-groups/Group_3_2"
df = spark.read.parquet(f"dbfs:/mnt/mids-w261/OTPW_60M_Backup/")

In [0]:
display(df)

#Simple Data Metrics

In [0]:
# rows and cols
print(f'# of Columns: {len(df.columns)}')
print(f'# of Rows: {df.count()}')


In [0]:
# size of data
size_bytes = sum(file.size for file in dbutils.fs.ls("dbfs:/mnt/mids-w261/OTPW_60M_Backup/"))
size_gb = size_bytes / (1024**3)

print(f"{size_gb} GB")

In [0]:
# schema
df.printSchema()

In [0]:
# of cancelled flights, we will remove the cancelled flights from our analysis
df.select((F.avg(F.col("CANCELLED").cast("double")) * 100).alias("cancelled_pct")).display()

In [0]:
# missing values
total = df.count()

missing_row = df.select([
    (F.count(F.when(F.col(c).isNull(), c)) / total * 100).alias(c)
    for c in df.columns
])

missing_long = missing_row.selectExpr(
    "stack({0}, {1}) as (column, missing_pct)".format(
        len(df.columns),
        ", ".join([f"'{c}', `{c}`" for c in df.columns])
    )
)

display(missing_long.orderBy(F.desc("missing_pct")))

In [0]:
high_missing = missing_long.filter(F.col("missing_pct") > 50)

display(high_missing.orderBy(F.desc("missing_pct")))

In [0]:
high_missing_count = high_missing.count()

print(f'Count: {high_missing_count}')
print(f'Percent: {high_missing_count/len(df.columns)}')

In [0]:
# filter out cancelled flights
df = df.filter(F.col("CANCELLED").cast("double") == 0)

# filter for only FM-15 report type
df = df.filter(F.col("REPORT_TYPE") == "FM-15")

# filter missing departure delay
df= df.filter(
    F.col("DEP_DEL15").isNotNull() & (~F.isnan(F.col("DEP_DEL15")))
)

In [0]:
plot_df = (
    df.select(F.col("DEP_DEL15").cast("double").alias("DEP_DEL15"))
      .filter(F.col("DEP_DEL15").isNotNull())
      .groupBy("DEP_DEL15")
      .count()
      .orderBy("DEP_DEL15")
      .toPandas()
)

# map labels
plot_df["label"] = plot_df["DEP_DEL15"].map({
    0.0: "On-time",
    1.0: "Delayed"
})

plt.figure(figsize=(6,4))
plt.bar(plot_df["label"], plot_df["count"])

plt.xlabel("Flight Status")
plt.ylabel("Number of Flights")
plt.title("Flight Delay Distribution (DEP_DEL15)")

plt.show()

In [0]:
# -------------------------
# Feature engineering
# -------------------------
df2 = df.withColumn(
    "hour",
    (F.col("CRS_DEP_TIME").cast("int") / 100).cast("int")
).withColumn(
    "DEP_DEL15",
    F.col("DEP_DEL15").cast("double")
).withColumn(
    "DAY_OF_WEEK",
    F.col("DAY_OF_WEEK").cast("int")
).withColumn(
    "MONTH",
    F.col("MONTH").cast("int")
)

# -------------------------
# Aggregations
# -------------------------
hour_df = df2.groupBy("hour").agg(F.avg("DEP_DEL15").alias("delay_rate")).orderBy("hour")
dow_df = df2.groupBy("DAY_OF_WEEK").agg(F.avg("DEP_DEL15").alias("delay_rate")).orderBy("DAY_OF_WEEK")
month_df = df2.groupBy("MONTH").agg(F.avg("DEP_DEL15").alias("delay_rate")).orderBy("MONTH")

# -------------------------
# Pandas
# -------------------------
hour_pd = hour_df.toPandas()
dow_pd = dow_df.toPandas()
month_pd = month_df.toPandas()

# -------------------------
# Plot
# -------------------------
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Hour
axes[0].plot(hour_pd["hour"], hour_pd["delay_rate"], marker="o")
axes[0].set_title("Delay Rate by Hour of Day")
axes[0].set_xlabel("Hour")
axes[0].set_xticks(hour_pd["hour"])

# Day of Week
axes[1].plot(dow_pd["DAY_OF_WEEK"], dow_pd["delay_rate"], marker="o")
axes[1].set_title("Delay Rate by Day of Week")
axes[1].set_xlabel("Day of Week")
axes[1].set_xticks(dow_pd["DAY_OF_WEEK"])

# Month
axes[2].plot(month_pd["MONTH"], month_pd["delay_rate"], marker="o")
axes[2].set_title("Delay Rate by Month")
axes[2].set_xlabel("Month")
axes[2].set_xticks(month_pd["MONTH"])

plt.tight_layout()
plt.show()

In [0]:
# ---------------- Clean delay column ----------------
df2 = df.withColumn("DEP_DEL15", F.col("DEP_DEL15").cast("double"))

# ---------------- Carrier (TOP 10 by volume-aware delay rate) ----------------
carrier_df = (
    df2.groupBy("OP_UNIQUE_CARRIER")
       .agg(
           F.avg("DEP_DEL15").alias("delay_rate"),
           F.count("*").alias("num_flights")
       )
       .orderBy("delay_rate", ascending=False)
)

carrier_pd = carrier_df.toPandas().head(10)

# ---------------- Origin (filter low-volume airports) ----------------
origin_df = (
    df2.groupBy("ORIGIN")
       .agg(
           F.avg("DEP_DEL15").alias("delay_rate"),
           F.count("*").alias("num_flights")
       )
       .filter(F.col("num_flights") > 1000)   # IMPORTANT FIX
       .orderBy("delay_rate", ascending=False)
)

origin_pd = origin_df.toPandas().head(10)

# ---------------- Plot ----------------
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Carrier
axes[0].bar(carrier_pd["OP_UNIQUE_CARRIER"], carrier_pd["delay_rate"])
axes[0].set_title("Top 10 Airline Delay Rates")
axes[0].set_xlabel("Airline")
axes[0].set_ylabel("Delay Rate")
axes[0].tick_params(axis='x', rotation=45)

# Origin
axes[1].bar(origin_pd["ORIGIN"], origin_pd["delay_rate"])
axes[1].set_title("Top 10 Origin Airports (>1000 flights)")
axes[1].set_xlabel("Airport")
axes[1].set_ylabel("Delay Rate")
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [0]:
#TODO correlation heatmap